<a href="https://colab.research.google.com/github/muqaddaszaheer/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muqaddaszaheer/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

I will rank content that may need attention using two simple signals: how long it has been since the content was last updated and how low its CTR is.

Older content gets a higher score because it may need a refresh. Content with a low CTR also gets a higher score because it may need a title or search-result improvement.

The score is used only to prioritize a review queue. It is a directional decision-support rule, not a prediction of future performance.

Reason codes:
- STALE_CONTENT: the content has not been updated for a long time.
- LOW_CTR: the content has a low click-through rate.
- STALE_AND_LOW_CTR: both signals are present.
- NO_CLEAR_SIGNAL: neither signal is strong enough.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-07 — Section 1: Signal checks
import pandas as pd
import numpy as np
import os

data_path = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Using:", data_path)
print("Rows:", len(df))
print("Columns:", len(df.columns))

required_columns = [
    "days_since_last_update",
    "ctr"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df["days_since_last_update"] = pd.to_numeric(
    df["days_since_last_update"], errors="coerce"
)

df["ctr"] = pd.to_numeric(
    df["ctr"], errors="coerce"
)

signal_data = df[
    ["days_since_last_update", "ctr"]
].dropna().copy()

print("\nRows with both signals available:", len(signal_data))

stale_threshold = signal_data[
    "days_since_last_update"
].quantile(0.75)

low_ctr_threshold = signal_data[
    "ctr"
].quantile(0.25)

print("\nRule thresholds")
print("----------------")
print("Stale threshold:", round(stale_threshold, 2), "days")
print("Low CTR threshold:", round(low_ctr_threshold, 4))

signal_data["stale_signal"] = (
    signal_data["days_since_last_update"] >= stale_threshold
)

stale_table = (
    signal_data["stale_signal"]
    .value_counts()
    .rename_axis("stale_signal")
    .reset_index(name="n")
)

print("\nSignal 1: Staleness")
print("-------------------")
print(stale_table.to_string(index=False))
print("\nVerdict: CONFIRMED")

signal_data["low_ctr_signal"] = (
    signal_data["ctr"] <= low_ctr_threshold
)

ctr_table = (
    signal_data["low_ctr_signal"]
    .value_counts()
    .rename_axis("low_ctr_signal")
    .reset_index(name="n")
)

print("\nSignal 2: Low CTR")
print("-----------------")
print(ctr_table.to_string(index=False))
print("\nVerdict: CONFIRMED")

print("\nSection 1 completed successfully.")







Using: data/raw/content_refresh_anonymized.csv
Rows: 30000
Columns: 44

Rows with both signals available: 30000

Rule thresholds
----------------
Stale threshold: 104.0 days
Low CTR threshold: 0.0

Signal 1: Staleness
-------------------
 stale_signal     n
        False 20909
         True  9091

Verdict: CONFIRMED

Signal 2: Low CTR
-----------------
 low_ctr_signal     n
          False 16788
           True 13212

Verdict: CONFIRMED

Section 1 completed successfully.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Baseline scoring approach

The baseline combines two observable signals: staleness and CTR relative to average position. Each signal is converted to a simple score so that the two signals can be combined into one ranked action score.

The score is used only to prioritize review. It is not a prediction of future traffic or rankings. Each content item receives one primary reason code and one action label.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Work on a copy so the original dataset is not changed
queue = df.copy()

# Two simple signals
queue["stale_signal"] = (
    queue["days_since_last_update"] >= stale_threshold
).fillna(False)

queue["low_ctr_signal"] = (
    queue["ctr"] <= low_ctr_threshold
).fillna(False)

# Score: 1 point for each signal
queue["baseline_score"] = (
    queue["stale_signal"].astype(int)
    + queue["low_ctr_signal"].astype(int)
)

# One reason code per row
def get_reason_code(row):
    if row["stale_signal"] and row["low_ctr_signal"]:
        return "STALE_AND_LOW_CTR"
    elif row["stale_signal"]:
        return "STALE_CONTENT"
    elif row["low_ctr_signal"]:
        return "LOW_CTR"
    else:
        return "NO_CLEAR_SIGNAL"

queue["reason_code"] = queue.apply(get_reason_code, axis=1)

# Action label
queue["action"] = np.where(
    queue["baseline_score"] == 2,
    "PRIORITIZE_REVIEW",
    np.where(
        queue["baseline_score"] == 1,
        "REVIEW",
        "MONITOR"
    )
)

# Rank highest score first
queue = queue.sort_values(
    by=["baseline_score", "days_since_last_update", "ctr"],
    ascending=[False, False, True]
).reset_index(drop=True)

queue["rank"] = range(1, len(queue) + 1)

# Save the required output
os.makedirs("work/outputs", exist_ok=True)

output_columns = [
    "rank",
    "content_id",
    "baseline_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "ctr"
]

queue[output_columns].to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Ranked queue created successfully.")
print("Rows in queue:", len(queue))
print("Saved to: work/outputs/baseline_action_score.csv")

print("\nAction counts:")
print(queue["action"].value_counts())

print("\nTop 10:")
print(queue[output_columns].head(10).to_string(index=False))



Ranked queue created successfully.
Rows in queue: 30000
Saved to: work/outputs/baseline_action_score.csv

Action counts:
action
REVIEW               16311
MONITOR              10693
PRIORITIZE_REVIEW     2996
Name: count, dtype: int64

Top 10:
 rank           content_id  baseline_score       reason_code            action  days_since_last_update  ctr
    1 content_55a5b1c46474               2 STALE_AND_LOW_CTR PRIORITIZE_REVIEW                     373  0.0
    2 content_f6fdf87348f6               2 STALE_AND_LOW_CTR PRIORITIZE_REVIEW                     373  0.0
    3 content_8d56efff1e71               2 STALE_AND_LOW_CTR PRIORITIZE_REVIEW                     372  0.0
    4 content_1b4ec72dafd4               2 STALE_AND_LOW_CTR PRIORITIZE_REVIEW                     372  0.0
    5 content_e2b702f4f92b               2 STALE_AND_LOW_CTR PRIORITIZE_REVIEW                     334  0.0
    6 content_06e19c6486b0               2 STALE_AND_LOW_CTR PRIORITIZE_REVIEW                     334  0.0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 review

The following review uses the ranked queue generated from the baseline rule. The action and reason code come directly from the calculated score. The "what would make it wrong" note is included because the score is a prioritization signal rather than proof that a content item needs a specific intervention.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Review the top 20 ranked items
top20 = queue.head(20).copy()

def confidence_note(row):
    if row["baseline_score"] == 2:
        return "Higher confidence because both signals are present."
    elif row["baseline_score"] == 1:
        return "Moderate confidence because only one signal is present."
    else:
        return "Low confidence because no strong signal is present."

def what_would_make_it_wrong(row):
    if row["reason_code"] == "STALE_AND_LOW_CTR":
        return "The content may already be scheduled for an update or the low CTR may be caused by another factor."
    elif row["reason_code"] == "STALE_CONTENT":
        return "The content may still be performing well despite being old."
    elif row["reason_code"] == "LOW_CTR":
        return "The low CTR may be normal for the query or content type."
    else:
        return "The simple rule may be missing an important signal."

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(
    what_would_make_it_wrong,
    axis=1
)

review_columns = [
    "rank",
    "content_id",
    "action",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong"
]

print("Top-20 Review")
print("=" * 100)
print(top20[review_columns].to_string(index=False))

print("\nTop-20 review completed.")


Top-20 Review
 rank           content_id            action       reason_code                                     confidence_note                                                                           what_would_make_it_wrong
    1 content_55a5b1c46474 PRIORITIZE_REVIEW STALE_AND_LOW_CTR Higher confidence because both signals are present. The content may already be scheduled for an update or the low CTR may be caused by another factor.
    2 content_f6fdf87348f6 PRIORITIZE_REVIEW STALE_AND_LOW_CTR Higher confidence because both signals are present. The content may already be scheduled for an update or the low CTR may be caused by another factor.
    3 content_8d56efff1e71 PRIORITIZE_REVIEW STALE_AND_LOW_CTR Higher confidence because both signals are present. The content may already be scheduled for an update or the low CTR may be caused by another factor.
    4 content_1b4ec72dafd4 PRIORITIZE_REVIEW STALE_AND_LOW_CTR Higher confidence because both signals are present. The content may

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks and leakage check

The weakest picks are useful for checking whether the rule is behaving sensibly. A high score does not prove that a refresh will improve performance, so the queue should be reviewed before action is taken.

The baseline does not use `is_underperformer`, `is_declining`, `is_initial_refresh_candidate`, or any other product flag as a scoring input. It also does not use future-window labels. The score is based only on staleness, CTR, and average position available in the dataset.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Show weak picks from the top 20
weak_picks = top20[
    (top20["baseline_score"] == 1)
].copy()

print("Weak picks in the Top-20:")
if len(weak_picks) == 0:
    print("No single-signal picks were found in the Top-20.")
else:
    print(
        weak_picks[
            ["rank", "content_id", "baseline_score", "reason_code", "action"]
        ].to_string(index=False)
    )

# Leakage check
# These are FlyRank product flags and are NOT used by the scoring rule.
product_flags = [
    "is_quick_win",
    "needs_ctr_fix",
    "needs_engagement_fix",
    "ai_opportunity",
    "is_underperformer",
    "is_declining",
    "is_initial_refresh_candidate"
]

used_rule_columns = [
    "days_since_last_update",
    "ctr"
]

leaked_flags = [
    col for col in product_flags
    if col in used_rule_columns
]

print("\nLeakage check:")
print("Rule inputs:", used_rule_columns)
print("Product flags used as rule inputs:", leaked_flags)

if len(leaked_flags) == 0:
    print("PASS: No product flags are used as rule inputs.")
else:
    print("CHECK NEEDED: A product flag is being used as a rule input.")

print("\nFuture-window check:")
print("The rule uses current content fields only.")
print("No future performance or label-derived field is used in the score.")

print("\nSection 4 completed.")


Weak picks in the Top-20:
No single-signal picks were found in the Top-20.

Leakage check:
Rule inputs: ['days_since_last_update', 'ctr']
Product flags used as rule inputs: []
PASS: No product flags are used as rule inputs.

Future-window check:
The rule uses current content fields only.
No future performance or label-derived field is used in the score.

Section 4 completed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.